In [1]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy import stats as spstats
import numpy as np
import sys
from sklearn.model_selection import StratifiedKFold
import os
import warnings
warnings.filterwarnings('ignore')
sys.path.append("_libs")
from select_features import FC_DimRed
from utils import load_cov_mats, load_atms, load_corr_mats
import pandas as pd

## 

In [3]:
while os.getcwd().split(os.sep)[-1] != "REDDI":
    os.chdir("../")
project_root = os.getcwd()  # should be .../REDDI

from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

datasets = {
    "ATM": load_atms(zscore=1.6),
    "Covariance": load_cov_mats(),
    "Correlation": load_corr_mats(),
}

rankings = {}

for name, (X, y) in datasets.items():

    X = X[:, :78, :78]
    rankings[name] = []

    print(f"\n{name}")

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):

        selector = FC_DimRed(eta_threshold=0.1, nb_nodes=78)
        selector.fit_transform(X[train_idx], y[train_idx], metric="eta-squared")

        rankings[name].append(selector.node_select_)

        print(f"Fold {fold}: {selector.node_select_}")

pd_rankings = pd.DataFrame.from_dict(rankings, orient="index")

Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)
Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)
Shape of all data: (109, 116, 116)
Shape of target data: (109,)
Shape of all mapped target data (0, 1, 2, 3): (109,)

ATM
Fold 1: [36 54 51 64 57 56 52 25 38 47 14 11 15 46 17 77  1 66 65 18 53 58 63 67
 28 55 60  6 12 61 24 62 49  3  8 73 29 19 20 23  7 13 59 27 37 42 10 45
 40 26 74 75 50 35 68 16  5 21  9  4 30 72 41 76 31 22 71 70 69 33 39 43
 48  2 32 34 44  0]
Fold 2: [36 64 49 15 38 25 63  6 58 56 17 54 59 65 75 57 52 18 51 76 14 20 50 74
 37 11 61 32 77 12  1 19 10 21 60 35 43 42 13 16 45 28 29  8 62 66 22  9
  7 55 53 41 40 26 27 47 73 72 71 67  5  0 24 46  3 31 23 70  2  4 48 68
 39 44 34 69 30 33]
Fold 3: [49 74 42 64 63 50  1 75 10 37 65 59 36 25  0 38 14 57 51 15 35 18 20 54
 17 56 52 21 12 24  6 28 76 47  3 53 27 11 66 16 60 61 67 58 45 77 26

In [4]:
import numpy as np
import pandas as pd

mean_ranks = {}
std_ranks = {}

for modality, folds in rankings.items():

    # matrice: righe = fold, colonne = nodi
    rank_matrix = np.zeros((len(folds), 78))

    for i, ranking in enumerate(folds):
        for rank, node in enumerate(ranking, start=1):
            rank_matrix[i, node] = rank

    mean_ranks[modality] = pd.Series(
        rank_matrix.mean(axis=0),
        index=np.arange(78),
        name=modality
    )
    std_ranks[modality] = pd.Series(
        rank_matrix.std(axis=0),
        index=np.arange(78),
        name=modality
    )

pd_mean_ranks = pd.DataFrame(mean_ranks)
pd_std_ranks = pd.DataFrame(std_ranks)

In [15]:
pd_std_ranks

,ATM,Covariance,Correlation
0,23.267144,1.720465,1.600000
1,7.863841,13.365628,21.611108
2,3.949684,2.227106,4.454211
3,15.628180,2.315167,3.249615
4,6.311894,8.518216,8.717798
...,...,...,...
73,11.822013,5.979967,10.665833
74,16.037456,19.005262,20.981897
75,15.704776,10.342147,9.239048
76,15.602564,14.634207,17.278889


In [20]:
ATM_rankings = []
COV_rankings = []
CORR_rankings = []
for i in range(5):
    ATM_rankings.append(pd_rankings.loc["ATM", i])
    COV_rankings.append(pd_rankings.loc["Covariance", i])
    CORR_rankings.append(pd_rankings.loc["Correlation", i])
pd_rankings[0]["ATM"]

array([36, 54, 51, 64, 57, 56, 52, 25, 38, 47, 14, 11, 15, 46, 17, 77,  1,
       66, 65, 18, 53, 58, 63, 67, 28, 55, 60,  6, 12, 61, 24, 62, 49,  3,
        8, 73, 29, 19, 20, 23,  7, 13, 59, 27, 37, 42, 10, 45, 40, 26, 74,
       75, 50, 35, 68, 16,  5, 21,  9,  4, 30, 72, 41, 76, 31, 22, 71, 70,
       69, 33, 39, 43, 48,  2, 32, 34, 44,  0], dtype=int64)

In [ ]:
how_many_times_selected_ATM = np.zeros((78))
for rank_list in ATM_rankings:
    for region,rank in enumerate(rank_list):
        if rank < 50:
            how_many_times_selected_ATM[region] += 1

how_many_times_selected_COV = np.zeros((78))
for rank_list in COV_rankings:
    for region,rank in enumerate(rank_list):
        if rank < 50:
            how_many_times_selected_COV[region] += 1

how_many_times_selected_CORR = np.zeros((78))
for rank_list in CORR_rankings:
    for region,rank in enumerate(rank_list):
        if rank < 50:
            how_many_times_selected_CORR[region] += 1
how_many_dict = {"ATM": how_many_times_selected_ATM, "Covariance": how_many_times_selected_COV, "Correlation": how_many_times_selected_CORR}

In [29]:
how_many_dict["Correlation"]

array([0., 0., 2., 3., 1., 1., 3., 4., 3., 3., 3., 4., 3., 4., 3., 3., 4.,
       4., 3., 0., 1., 3., 3., 3., 3., 3., 5., 3., 2., 4., 2., 3., 2., 2.,
       3., 4., 4., 4., 3., 4., 3., 3., 2., 3., 3., 4., 3., 2., 3., 4., 4.,
       4., 4., 4., 4., 5., 3., 4., 3., 5., 4., 3., 3., 4., 4., 4., 5., 2.,
       5., 4., 5., 4., 4., 2., 4., 5., 3., 3.])

In [6]:
ROI_AAL_list = np.array([ 'Rectus_L','Olfactory_L','Frontal_Sup_Orb_L','Frontal_Med_Orb_L','Frontal_Mid_Orb_L',
                'Frontal_Inf_Orb_L','Frontal_Sup_L','Frontal_Mid_L','Frontal_Inf_Oper_L','Frontal_Inf_Tri_L',
                'Frontal_Sup_Medial_L','Supp_Motor_Area_L','Paracentral_Lobule_L','Precentral_L','Rolandic_Oper_L',
                'Postcentral_L','Parietal_Sup_L','Parietal_Inf_L','SupraMarginal_L','Angular_L','Precuneus_L',
                'Occipital_Sup_L','Occipital_Mid_L','Occipital_Inf_L','Calcarine_L','Cuneus_L','Lingual_L',
                'Fusiform_L','Heschl_L','Temporal_Sup_L','Temporal_Mid_L','Temporal_Inf_L','Temporal_Pole_Sup_L',
                'Temporal_Pole_Mid_L','ParaHippocampal_L','Cingulum_Ant_L','Cingulum_Mid_L','Cingulum_Post_L',
                'Insula_L','Rectus_R','Olfactory_R','Frontal_Sup_Orb_R','Frontal_Med_Orb_R','Frontal_Mid_Orb_R',
                'Frontal_Inf_Orb_R','Frontal_Sup_R','Frontal_Mid_R','Frontal_Inf_Oper_R','Frontal_Inf_Tri_R',
                'Frontal_Sup_Medial_R','Supp_Motor_Area_R','Paracentral_Lobule_R','Precentral_R','Rolandic_Oper_R',
                'Postcentral_R','Parietal_Sup_R','Parietal_Inf_R', 'SupraMarginal_R','Angular_R','Precuneus_R',
                'Occipital_Sup_R','Occipital_Mid_R','Occipital_Inf_R','Calcarine_R','Cuneus_R','Lingual_R',
                'Fusiform_R','Heschl_R','Temporal_Sup_R','Temporal_Mid_R','Temporal_Inf_R','Temporal_Pole_Sup_R',
                'Temporal_Pole_Mid_R','ParaHippocampal_R','Cingulum_Ant_R','Cingulum_Mid_R','Cingulum_Post_R',
                'Insula_R','Hippocampus_L','Hippocampus_R','Amygdala_L','Amygdala_R','Caudate_L','Caudate_R',
                'Putamen_L','Putamen_R','Pallidum_L','Pallidum_R','Thalamus_L','Thalamus_R','Cerebelum_Crus1_L',
                'Cerebelum_Crus1_R','Cerebelum_Crus2_L','Cerebelum_Crus2_R','Cerebelum_3_L','Cerebelum_3_R',
                'Cerebelum_4_5_L','Cerebelum_4_5_R','Cerebelum_6_L','Cerebelum_6_R','Cerebelum_7b_L','Cerebelum_7b_R',
                'Cerebelum_8_L','Cerebelum_8_R','Cerebelum_9_L','Cerebelum_9_R','Cerebelum_10_L','Cerebelum_10_R',
                'Vermis_1_2','Vermis_3','Vermis_4_5','Vermis_6','Vermis_7','Vermis_8','Vermis_9','Vermis_10'])